In [ ]:
import json, os
import numpy as np
from collections import Counter
import random

import math

def make_rel_map_from_gt(gt):
    """
    Convert GT ranking into graded relevance:
    first item gets len(gt), last gets 1.
    """
    n = len(gt)
    return {item: n - i for i, item in enumerate(gt)}  # 8..1

def dcg_at_k(ranking, k, rel_map, gain="exp"):
    """
    DCG@k with either:
      - exp gain: 2^rel - 1  (default in IR)
      - linear gain: rel
    Items not in rel_map get rel=0.
    """
    s = 0.0
    for i, item in enumerate(ranking[:k], start=1):
        rel = rel_map.get(item, 0)
        if gain == "exp":
            g = (2 ** rel) - 1
        elif gain == "linear":
            g = rel
        else:
            raise ValueError("gain must be 'exp' or 'linear'")
        s += g / math.log2(i + 1)
    return s

def ndcg_at_k(pred, gt, k, gain="exp"):
    if len(gt) == 0:
        return 0.0
    rel_map = make_rel_map_from_gt(gt)

    dcg = dcg_at_k(pred, k, rel_map, gain=gain)
    # Ideal ranking is GT order itself (already best-to-worst)
    idcg = dcg_at_k(gt, k, rel_map, gain=gain)

    return 0.0 if idcg == 0 else dcg / idcg

def create_ranking_from_counts(step_counts):
    """
    Create ranking from step counts.
    Sort by count (descending), then by step number (ascending) for ties.
    """
    # Convert to list of (step, count) tuples
    items = [(int(step), count) for step, count in step_counts.items()]
    # Sort by count (descending), then by step number (ascending)
    items_sorted = sorted(items, key=lambda x: (-x[1], x[0]))
    # Return list of step numbers in ranking order
    return [step for step, count in items_sorted]

def create_random_ranking(gt_ranking, seed=None):
    """
    Create a random ranking by shuffling the GT ranking.
    This serves as a baseline for comparison.
    """
    if len(gt_ranking) == 0:
        return []
    random_ranking = gt_ranking.copy()
    if seed is not None:
        random.seed(seed)
    random.shuffle(random_ranking)
    return random_ranking

In [ ]:
import glob

# Configuration
backbones = ['openai_gpt_4.1', 'openai_gpt_5.1', 'openai_o3_mini','together_openai_gpt_oss_120b', 'qwen_Qwen3_8B', 'claude_claude_sonnet_4_5']
tasks = ['manual', 'automatic']  # Final task groups for evaluation
seeds = [0,1,2]  # Adjust based on available seeds


# First, find files that exist in all three annotators (1, 2, 3) for each raw task
valid_files_by_raw_task = {}

for raw_task in tasks:
    valid_files = set()
    
    # Get files from annotator 1
    annotator1_path = f'annotated/1/{raw_task}'
    if os.path.exists(annotator1_path):
        files1 = set([f for f in os.listdir(annotator1_path) if f.endswith('.json')])
    else:
        files1 = set()
    
    # Get files from annotator 2
    annotator2_path = f'annotated/2/{raw_task}'
    if os.path.exists(annotator2_path):
        files2 = set([f for f in os.listdir(annotator2_path) if f.endswith('.json')])
    else:
        files2 = set()
    
    # Get files from annotator 3
    annotator3_path = f'annotated/3/{raw_task}'
    if os.path.exists(annotator3_path):
        files3 = set([f for f in os.listdir(annotator3_path) if f.endswith('.json')])
    else:
        files3 = set()
    
    # Find intersection: files that exist in all three annotators
    valid_files = files1 & files2 & files3
    valid_files_by_raw_task[raw_task] = valid_files
    print(f"Task {raw_task}: Found {len(valid_files)} files with all 3 annotators")

# Collect all results using only valid files
all_results = []

for backbone in backbones:
    for raw_task in tasks:
        valid_files = valid_files_by_raw_task.get(raw_task, set())
        if len(valid_files) == 0:
            continue
        
        # Map raw task to final task group
        final_task = raw_task
        
        for filename in valid_files:
            # Check if results exist for this file (using seed_0 as reference)
            # For cap, the path might be different, adjust if needed
            
            base_path = f'results/{backbone}/all_at_once_taxonomy/seed_0/{raw_task}'
            
            seed0_file = f'{base_path}/{filename}'
            
            if not os.path.exists(seed0_file):
                continue
            
            # Collect predictions from all seeds
            fail_step = []
            for seed in seeds:
                seed_file = seed0_file.replace('seed_0', f'seed_{seed}')
                if os.path.exists(seed_file):
                    try:
                        data = json.load(open(seed_file))
                        if 'failure_attribution' in data:
                            fail_step.extend([a['step_number'] for a in data['failure_attribution']])
                    except:
                        pass
            
            # Collect GT from all annotators (we know all 3 exist)
            fail_step_gt = []
            for annotator in [1, 2, 3]:
                gt_file = f'annotated/{annotator}/{raw_task}/{filename}'
                try:
                    data = json.load(open(gt_file))
                    for a in data['history']:
                        if a['fail_annotation'] == '1':
                            fail_step_gt.append(int(a['step']))
                except:
                    pass
            
            if len(fail_step) > 0 and len(fail_step_gt) > 0:
                all_results.append({
                    'backbone': backbone,
                    'task': final_task,  # Use final task group 
                    'raw_task': raw_task,  # Keep raw task for reference
                    'filename': filename,
                    'fail_step': fail_step,
                    'fail_step_gt': fail_step_gt,
                    'len_history': len(data['history'])
                })
            else:
                pass
                # print(f"Skipping {filename} of {backbone} and {raw_task} due to no failure steps")


print(f"\nProcessed {len(all_results)} files (only files with all 3 annotators)")

In [ ]:
# Calculate nDCG for each file and aggregate
all_ndcg_results = {
    'exp': {5: [], 300: []},
    'linear': {5: [], 300: []}
}

# Random baseline nDCG
random_ndcg_results = {
    'exp': {5: [], 300: []},
    'linear': {5: [], 300: []}
}

for result in all_results:
    fail_step = [a for a in result['fail_step'] if isinstance(a, int)]
    fail_step_gt = [a for a in result['fail_step_gt'] if isinstance(a, int)]
    
    if len(fail_step) == 0 or len(fail_step_gt) == 0:
        continue
    
    # Create rankings
    step_counts = Counter(fail_step)
    ranking_predict = create_ranking_from_counts(step_counts)
    
    step_counts_gt = Counter(fail_step_gt)
    ranking_gt = create_ranking_from_counts(step_counts_gt)
    
    # Create random ranking baseline
    ranking_random = create_random_ranking(random.sample(range(result['len_history']), min(len(ranking_predict), result['len_history'])))
    
    # Calculate nDCG for different k values
    for k in [5, 300]:
        for gain_type in ['exp', 'linear']:
            ndcg_val = ndcg_at_k(ranking_predict, ranking_gt, k, gain=gain_type)
            all_ndcg_results[gain_type][k].append(ndcg_val)
            
            # Random baseline
            random_ndcg_val = ndcg_at_k(ranking_random, ranking_gt, k, gain=gain_type)
            random_ndcg_results[gain_type][k].append(random_ndcg_val)


print("\n" + "=" * 60)
print("Breakdown by Backbone+Task")
print("=" * 60)

for backbone in backbones:
    for task in tasks:
        combo_results = [r for r in all_results if r['backbone'] == backbone and r['task'] == task]
        if len(combo_results) == 0:
            continue
        
        combo_ndcg = {
            'exp': {5: [], 300: []},
            'linear': {5: [], 300: []}
        }
        
        combo_random_ndcg = {
            'exp': {5: [], 300: []},
            'linear': {5: [], 300: []}
        }
        
        for result in combo_results:
            fail_step = [a for a in result['fail_step'] if isinstance(a, int)]
            fail_step_gt = [a for a in result['fail_step_gt'] if isinstance(a, int)]
            
            if len(fail_step) == 0 or len(fail_step_gt) == 0:
                continue
            
            step_counts = Counter(fail_step)
            ranking_predict = create_ranking_from_counts(step_counts)
            
            step_counts_gt = Counter(fail_step_gt)
            ranking_gt = create_ranking_from_counts(step_counts_gt)
            
            ranking_random = create_random_ranking(random.sample(range(result['len_history']), min(len(ranking_predict), result['len_history'])))
            
            for k in [5, 300]:
                for gain_type in ['exp', 'linear']:
                    ndcg_val = ndcg_at_k(ranking_predict, ranking_gt, k, gain=gain_type)
                    combo_ndcg[gain_type][k].append(ndcg_val)
                    
                    random_ndcg_val = ndcg_at_k(ranking_random, ranking_gt, k, gain=gain_type)
                    combo_random_ndcg[gain_type][k].append(random_ndcg_val)
        
        print(f"\n{backbone.upper()} + {task.upper()}:")
        for gain_type in ['exp', 'linear']:
            print(f"  {gain_type.upper()} gain:")
            for k in [5, 300]:
                if len(combo_ndcg[gain_type][k]) > 0:
                    avg_ndcg = np.mean(combo_ndcg[gain_type][k])
                    print(f"    nDCG@All = {avg_ndcg:.4f}" if k == 300 else f"    nDCG@{k} = {avg_ndcg:.4f}")
            
            print(f"  {gain_type.upper()} gain (RANDOM BASELINE):")
            for k in [5, 300]:
                if len(combo_random_ndcg[gain_type][k]) > 0:
                    avg_ndcg = np.mean(combo_random_ndcg[gain_type][k])
                    print(f"    nDCG@All = {avg_ndcg:.4f}" if k == 300 else f"    nDCG@{k} = {avg_ndcg:.4f}")
